In [107]:
# Milestone
# CDC dataset cleaning

In [108]:
import pandas as pd
import numpy as np

In [113]:
# Pull
file_path = "data/raw/cdc_wonder_maternal_mortality_2018_2023_raw.csv"
df = pd.read_csv(file_path)

In [116]:
# standarize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)


In [118]:
# first clean up step : drop unnecessary columns
df = df.drop(
    columns = [
        "notes", 
        "state_code",
        "year_code",
        "single_race_6_code",
        "crude_rate"
    ]
)

In [120]:
# rename column
df = df.rename(columns = {"single_race_6": "race"})

In [121]:
# convert numeric columns safely (suppressed -> NaN, 0 = 0, # = #'s)
for col in ["year", "deaths", "population"]:
    df[col] = pd.to_numeric(df[col], errors = "coerce")

In [122]:
# drop NA's but keep supressed and zero values
df = df.dropna(subset = ["year"]).copy()
df.loc[:, "year"] = df["year"].astype(int)

In [123]:
# verify suppression and zeroes coexist correctly
df["deaths"].value_counts(dropna = False).head(10)

deaths
NaN     242
0.0      86
11.0     19
15.0     17
13.0     17
12.0     16
10.0     14
14.0     14
20.0     13
18.0     12
Name: count, dtype: int64

In [125]:
mortality_long = df[df["year"].between(2021, 2023)].copy()

In [126]:
mortality_long.to_csv(
    "data/clean/cdc_wonder_maternal_mortality_2021_2023_clean.csv",
    index=False
)

cdc_wonder_maternal_mortality_2021_2023_clean.csv
- Source: CDC WONDER Multiple Cause of Death
- ICD-10: O00–O99
- Race: Single Race 6 (Black, White)
- Years: 2021–2023
- Suppressed values retained as NA

In [127]:
# aggregate deaths and population to mitigate suppression
state_race_totals = (
    mortality_long
    .groupby(["state", "race"], as_index=False)
    .agg(
        deaths_total=("deaths", "sum"),
        population_total=("population", "sum")
    )
)

state_race_totals["mortality_rate_per_100k"] = (
    state_race_totals["deaths_total"]
    / state_race_totals["population_total"]
) * 100_000


In [128]:
# pivot to wide format for racial disparity calculations
state_level_rates_wide = (
    state_race_totals
    .pivot_table(
        index="state",
        columns="race",
        values="mortality_rate_per_100k"
    )
    .reset_index()
)

state_level_rates_wide = state_level_rates_wide.rename(columns={
    "Black or African American": "mortality_rate_black",
    "White": "mortality_rate_white"
})


In [129]:
# compute disparity metrics
state_level_rates_wide["gap_black_minus_white"] = (
    state_level_rates_wide["mortality_rate_black"]
    - state_level_rates_wide["mortality_rate_white"]
)

state_level_rates_wide["ratio_black_to_white"] = (
    state_level_rates_wide["mortality_rate_black"]
    / state_level_rates_wide["mortality_rate_white"]
)


In [130]:
# assess analytic coverage
state_level_rates_wide["has_black_and_white"] = (
    state_level_rates_wide[[
        "mortality_rate_black",
        "mortality_rate_white"
    ]]
    .notna()
    .all(axis=1)
)

state_disparities_complete = (
    state_level_rates_wide
    .query("has_black_and_white")
    .copy()
)


In [131]:
# save canonical outputs
state_race_totals.to_csv(
    "data/analysis/state_race_mortality_rates_2021_2023.csv",
    index=False
)

state_disparities_complete.to_csv(
    "data/analysis/state_maternal_racial_disparities_2021_2023.csv",
    index=False
)

`analysis_df` contains state-level maternal mortality rates for Black and White women (aggregated across 2021–2023), along with absolute and relative racial disparity metrics. This dataset is intended for direct use in downstream analyses and for merging with hospital characteristics data.

In [132]:
# final analysis dataset
analysis_df = state_disparities_complete.copy()


In [133]:
# summary statistics (descriptive context)
analysis_df[[
    "mortality_rate_black",
    "mortality_rate_white",
    "gap_black_minus_white",
    "ratio_black_to_white"
]].describe()


race,mortality_rate_black,mortality_rate_white,gap_black_minus_white,ratio_black_to_white
count,51.000000,51.000000,51.000000,34.000000
mean,0.384693,0.196240,0.188453,1.860558
std,0.520093,0.170387,0.423856,1.788140
min,0.000000,0.000000,-0.441519,0.000000
25%,0.000000,0.000000,-0.117563,0.000000
50%,0.000000,0.208902,0.000000,2.085840
75%,0.829993,0.301990,0.550870,3.075970
max,1.580804,0.543346,1.147326,6.141576


In [134]:
# Summarize by state (if you want geographic emphasis)
analysis_df.groupby("state")[[
    "gap_black_minus_white",
    "ratio_black_to_white"
]].describe()


race                 gap_black_minus_white                                    \
                                     count      mean std       min       25%   
state                                                                          
Alabama                                1.0  0.920132 NaN  0.920132  0.920132   
Alaska                                 1.0  0.000000 NaN  0.000000  0.000000   
Arizona                                1.0 -0.337950 NaN -0.337950 -0.337950   
Arkansas                               1.0  0.267600 NaN  0.267600  0.267600   
California                             1.0  0.430869 NaN  0.430869  0.430869   
Colorado                               1.0 -0.277927 NaN -0.277927 -0.277927   
Connecticut                            1.0 -0.129323 NaN -0.129323 -0.129323   
Delaware                               1.0  0.000000 NaN  0.000000  0.000000   
District of Columbia                   1.0  0.000000 NaN  0.000000  0.000000   
Florida                                1.0  0.722963 NaN  0.722963  0.722963   
Georgia                                1.0  0.753162 NaN  0.753162  0.753162   
Hawaii                                 1.0  0.000000 NaN  0.000000  0.000000   
Idaho                                  1.0  0.000000 NaN  0.000000  0.000000   
Illinois                               1.0  0.642482 NaN  0.642482  0.642482   
Indiana                                1.0 -0.441519 NaN -0.441519 -0.441519   
Iowa                                   1.0 -0.185507 NaN -0.185507 -0.185507   
Kansas                                 1.0  0.000000 NaN  0.000000  0.000000   
Kentucky                               1.0 -0.433336 NaN -0.433336 -0.433336   
Louisiana                              1.0  1.074084 NaN  1.074084  1.074084   
Maine                                  1.0  0.000000 NaN  0.000000  0.000000   
Maryland                               1.0  0.262995 NaN  0.262995  0.262995   
Massachusetts                          1.0 -0.138109 NaN -0.138109 -0.138109   
Michigan                               1.0  0.430721 NaN  0.430721  0.430721   
Minnesota                              1.0 -0.105803 NaN -0.105803 -0.105803   
Mississippi                            1.0  1.147326 NaN  1.147326  1.147326   
Missouri                               1.0  0.197985 NaN  0.197985  0.197985   
Montana                                1.0  0.000000 NaN  0.000000  0.000000   
Nebraska                               1.0  0.000000 NaN  0.000000  0.000000   
Nevada                                 1.0  0.000000 NaN  0.000000  0.000000   
New Hampshire                          1.0  0.000000 NaN  0.000000  0.000000   
New Jersey                             1.0  0.640168 NaN  0.640168  0.640168   
New Mexico                             1.0 -0.272249 NaN -0.272249 -0.272249   
New York                               1.0  0.609465 NaN  0.609465  0.609465   
North Carolina                         1.0  0.812192 NaN  0.812192  0.812192   
North Dakota                           1.0  0.000000 NaN  0.000000  0.000000   
Ohio                                   1.0  0.790149 NaN  0.790149  0.790149   
Oklahoma                               1.0 -0.261135 NaN -0.261135 -0.261135   
Oregon                                 1.0 -0.256311 NaN -0.256311 -0.256311   
Pennsylvania                           1.0  0.516816 NaN  0.516816  0.516816   
Rhode Island                           1.0  0.000000 NaN  0.000000  0.000000   
South Carolina                         1.0  0.584924 NaN  0.584924  0.584924   
South Dakota                           1.0  0.000000 NaN  0.000000  0.000000   
Tennessee                              1.0  1.037458 NaN  1.037458  1.037458   
Texas                                  1.0  0.485958 NaN  0.485958  0.485958   
Utah                                   1.0 -0.142459 NaN -0.142459 -0.142459   
Vermont                                1.0  0.000000 NaN  0.000000  0.000000   
Virginia                               1.0  0.680511 NaN  0.680511  0.680511   
